# Credit Risk Analysis

In [5]:
pip install imblearn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install icecream


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Importation des librairies
import pandas as pd
import seaborn as sns
import numpy as np
from icecream import ic
from pandas import read_csv
from pandas import set_option

from matplotlib import pyplot as plt

from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTENC
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline 

from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import  GradientBoostingClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import auc
from sklearn.metrics import precision_score
from sklearn.metrics import make_scorer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate



## Collecte initiale des données

In [ ]:
# Collecte des données à analyser
credit_filename = 'data/credit.csv'

try:
    credit_df = read_csv(credit_filename)
    print("L'importation des données de crédit à bien fonctionné")
except Exception as e:
    print("Une erreur est survenue dans l'importation des données de crédit")

credit_df_copy = credit_df.copy()


## Credit.csv

### Description des données de credit.csv

In [ ]:
# Affichages des informations du dataset
credit_df_copy.info()


In [ ]:
#Affichage du nombre de valeurs égale à '?' par attribut dans le dataset credit.csv
(credit_df_copy == '?').sum()

In [ ]:
# affichage des valeurs manquantes par attribut dans le dataset credit.csv
credit_df_copy.isnull().sum()

### Prétraitement crédit.csv

#### Prétraitement de 'Industry'

In [ ]:
#on va utiliser un encodage onehot etant donnée qu'il n'y a pas d'ordre particulier
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_Industry = encoder.fit_transform(credit_df_copy[['Industry']])
df_encoded_Industry = pd.DataFrame(encoded_Industry, columns=encoder.get_feature_names_out(['Industry']))


#### Prétraitement de 'Ethnicity'

In [ ]:
#on va utiliser un encodage onehot etant donnée qu'il n'y a pas d'ordre particulier
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_Ethnicity = encoder.fit_transform(credit_df_copy[['Ethnicity']])
df_encoded_Ethnicity = pd.DataFrame(encoded_Ethnicity, columns=encoder.get_feature_names_out(['Ethnicity']))

#### Prétraitement de 'Citizen'

In [ ]:
#on va utiliser un encodage onehot etant donnée qu'il n'y a pas d'ordre particulier
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_Citizen = encoder.fit_transform(credit_df_copy[['Citizen']])
df_encoded_Citizen = pd.DataFrame(encoded_Citizen, columns=encoder.get_feature_names_out(['Citizen']))

#### Prétraitement de 'Income'

In [ ]:
# On va vérifier le nombre d'entrée pareil
credit_df_copy.groupby('Income').size().head(20)

In [ ]:
#Vérifer l'impact des revenus égale à 0 sur l'approbation
credit_df_copy.groupby((credit_df_copy['Income'] == 0))['Approved'].mean()
#Il n'y a pas vraiment de difference dans le taux de refus/acceptation pour une personne avec zéro comme income vs une personne avec plus de zéro comme income ce qui ne fais pas de sens. Il y a probablement erreur lors de la saisie.


In [ ]:
#Vérifier l'impact entre le revenu égale à 0 et l'attribut employed
credit_df.groupby((credit_df['Income'] == 0))['Employed'].mean()
#Dans 80% des cas quand le income est à 0 les individu sont sans emploi et dans 20% ils ont un emploi

In [ ]:
#verifier l'impact entre le revenu égale à 0 et l'attribut YearsEmployed
mean_worked_years_when_unemployed_and_no_income =credit_df.loc[(credit_df['Income'] == 0) & (credit_df['Employed'] == 0)]['YearsEmployed'].mean().round(2)
print(f"Lorsque l'individu n'a pas d'emploie et n'a pas de revenu, il a travailler en moyenne {mean_worked_years_when_unemployed_and_no_income} années")
# On valide les nombreux cas ou les individus n'ont pas de revenue.

In [ ]:
#Vérifer l'impact des revenus égale à 1 sur l'approbation
credit_df_copy.groupby((credit_df_copy['Income'] == 1))['Approved'].mean()
#pres de 100% des revenue egale a 1 sont des refusé. 

##### Y a t'il des cas ou le revenue est égale a zero et l'individu est employé et accumule des années d'emploi?

In [ ]:
nb_person_no_income_employed_with_years = credit_df_copy.loc[(credit_df_copy['Income'] == 0) & (credit_df_copy['Employed'] == 1) & (credit_df_copy['YearsEmployed'] != 0)]['Approved'].count()
print(f"Il existe {nb_person_no_income_employed_with_years} entrées dans le dataset de personnes employée depuis une période non nulle qui ont un income qui est de zéro.\nCes entrées doivent être traitées car il s'agit d'erreurs.\n")
print(f"On remplace ces valeurs par NaN et on les traites avec un SimpleImputer avec une stratégie de médiande")

In [ ]:
#On remplace les valeurs de Income ciblé par nan
credit_df_copy['Income'] = credit_df_copy['Income'].astype(float)
credit_df_copy.loc[(credit_df_copy['Income'] == 0) & (credit_df_copy['Employed'] == 1) & (credit_df_copy['YearsEmployed'] != 0), 'Income' ] = credit_df_copy.loc[(credit_df_copy['Income'] == 0) & (credit_df_copy['Employed'] == 1) & (credit_df_copy['YearsEmployed'] != 0), 'Income' ].replace(0,np.nan)


In [ ]:
ligne_revenu_cible = credit_df_copy.loc[credit_df_copy['Income'].isnull()].head(10)
index_revenu_cible  = ligne_revenu_cible.index
ligne_revenu_cible['Income']

In [ ]:
#On remplace maintenant les occurences de nan avec notre fonction de imputation 
imp_income = SimpleImputer(strategy='median')
credit_df_copy['Income'] = imp_income.fit_transform(credit_df_copy[['Income']])

In [ ]:
credit_df_copy.loc[index_revenu_cible]['Income']

In [ ]:
#Concaténation du df final de crédit suite au prétraitement
credit_df_final =pd.concat([credit_df_copy.iloc[:,:5], df_encoded_Industry, df_encoded_Ethnicity, credit_df_copy.iloc[:,7:12], df_encoded_Citizen, credit_df_copy.iloc[:,14:]], axis=1)
credit_df_final
credit_df_final.to_csv("data/processed/credit_df_final.csv", index=False)

### Statistique descriptives de credit.csv

##### Comment sont répartis les attributs numériques dans le dataset?

In [ ]:
print(f"Statistique descriptive des attributs numérique\n------------------------------------------\n{credit_df_copy.describe().round(2)}")
#credit_df_copy.describe().round(2)

In [ ]:
#Affichage de la distribution des attributs du dataset
colonne_credit_numerique = credit_df_copy.select_dtypes(include=['int64','float64']).columns

for columns in colonne_credit_numerique:
    plt.figure(figsize=(10,5))
    sns.histplot(credit_df_copy[columns], kde=True)
    plt.title(f"Distribution de l'attribut {columns}")
    plt.show()

In [ ]:
# affichage des valeurs abérantes
colonne_credit_numerique = credit_df_copy.select_dtypes(include=['int64','float64']).columns

for columns in colonne_credit_numerique:
    plt.figure(figsize=(10,5))
    sns.boxplot(x = credit_df_copy[columns])
    

##### Comment sont répartis les attributs catégoriels dans le dataset?

In [ ]:
#print(f"Statistique descriptive des attributs non numérique\n-----------------------------------------\n{credit_df_copy[['Industry', 'Ethnicity', 'Citizen']].describe()}")
credit_df_copy[['Industry', 'Ethnicity', 'Citizen']].describe()

In [ ]:
# Affichons les categories des attributs catégoriels
colonne_categorielle = credit_df_copy.select_dtypes(include='object').columns
for columns in colonne_categorielle:
    df_category = credit_df_copy.groupby(columns).size()
    plt.figure(figsize=(10,5))
    df_category.plot.pie(autopct='%1.1f%%', title=f"repartition des {columns}")
    plt.show()

#### Age

##### Quelle est la corrélation entre l'age et le résultat de la demande de crédit?

In [ ]:
# Quelle est la corrélation entre l'age et l'approbation de crédit
matrice_de_correlation_age = credit_df_copy[['Age', 'Approved']]
corr_age = matrice_de_correlation_age.copy().corr()
print(corr_age)
plt.figure(figsize=(6,4))
sns.heatmap(corr_age, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des ages et de l'approbation de crédit")
plt.show()

#### Debt

##### Quelle est la corrélation entre les dettes et le résultat de la demande de crédit?

In [ ]:
# Quelle est la corrélation entre les dettes et l'approbation de crédit
matrice_de_correlation_dette = credit_df_copy[['Debt', 'Approved']]
corr_debt = matrice_de_correlation_dette.copy().corr()
print(corr_debt)
plt.figure(figsize=(6,4))
sns.heatmap(corr_debt, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des dettes et de l'approbation de crédit")
plt.show()

#### YearsEmployed

##### Quelle est la corrélation entre le nombre d'années travaillé et le résultat de la demande de crédit?

In [ ]:
# Quelle est la corrélation entre les dettes et l'approbation de crédit
matrice_de_correlation_year_employed = credit_df_copy[['YearsEmployed', 'Approved']]
corr_years_employed = matrice_de_correlation_year_employed.copy().corr()
print(corr_years_employed)
plt.figure(figsize=(6,4))
sns.heatmap(corr_years_employed, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des années travaillé et de l'approbation de crédit")
plt.show()

#### ZipCode

##### Comment sont distribué les zipcode dans le dataset?

In [ ]:
# On va vérifier le nombre d'entrée pareil
entry_by_zip_code_sup_zero = credit_df_copy.loc[(credit_df_copy['ZipCode'] > 0)]
entry_by_zip_code_eq_zero = credit_df_copy.loc[(credit_df_copy['ZipCode'] == 0)]
nb_person_by_zip_code_sup_zero = entry_by_zip_code_sup_zero.groupby('ZipCode').size()
plt.figure(figsize=(8,6))
sns.histplot(nb_person_by_zip_code_sup_zero)
plt.title("Distribution des zipcode supérieur à zéro")
plt.xlabel("ZipCode")
plt.show()

nb_person_by_zip_code_eq_zero = entry_by_zip_code_eq_zero.groupby('ZipCode').size()
plt.figure(figsize=(8,6))
sns.barplot(nb_person_by_zip_code_eq_zero)
plt.title("Nombre d'entrée sans zipcode")
plt.show()

In [ ]:
number_zip_code_with_one_person = (credit_df_copy.groupby('ZipCode').size() == 1).sum() 
number_zip_code_not_with_one_person = (credit_df_copy.groupby('ZipCode').size() != 1).sum()
print(f"Il y a {number_zip_code_with_one_person} categorie de zipcode dans le dataset avec un seul individu ")


##### Quelle est la corrélation entre le zipcode et le résultat de la demande de crédit?

In [ ]:
# A explorer mais on peut se poser s'il y a une reelle corelation entre le zipcode et le resultat de l'approbation. Avec si peu de donnée pour certain zipcode 1 seule par zipcode pourra t'on predire avec le zipcode?

In [ ]:
# Quelle est la corrélation entre le zipcode et l'approbation
matrice_de_correlation = credit_df_copy[['ZipCode', 'Approved']]
corr = matrice_de_correlation.copy().corr()
corr

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des zipcode et de l'approbation de crédit")
plt.show()

#### Income

##### Quelle est la corrélation entre le income et le résultat de la demande de crédit?

In [ ]:
# Quelle est la corrélation entre le income imputé et l'approbation
matrice_de_correlation_income = credit_df_copy[['Income', 'Approved']]
corr_income = matrice_de_correlation_income.copy().corr()
print(corr_income)

plt.figure(figsize=(6,4))
sns.heatmap(corr_income, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des incomes et de l'approbation de crédit")
plt.show()

In [ ]:
# Quelle est la corrélation entre le income initial et l'approbation
matrice_de_correlation_income_initial = credit_df[['Income', 'Approved']]
corr_income_initial = matrice_de_correlation_income_initial.copy().corr()
print(corr_income_initial)

plt.figure(figsize=(6,4))
sns.heatmap(corr_income_initial, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des incomes initiaux et de l'approbation de crédit")
plt.show()

In [ ]:
#Au final l'attribut income n'est pas un attribut déterminant pour l'approbation de crédit

#### CreditScore

##### Comment sont distribué les valeurs de l'attribut CreditScore dans le dataset?

In [ ]:
# Nombre d'individu par creditscore
df_distribution_credit_score = credit_df_copy.groupby('CreditScore').size()
plt.figure(figsize=(8,5))
sns.barplot(df_distribution_credit_score)
plt.title("Distribution des resultats de creditscore du dataset")
plt.show()

##### Les individus avec un creditscore supérieur au maximum de la distribution boxplot(outliers) sont ils approuvés?

In [ ]:
# Analysons le outliers le plus extreme creditscore = 67
credit_df_copy.loc[credit_df_copy['CreditScore'] == 67]

In [ ]:
# Analysons les outliers supérieurs au maximum de la distribution boxplot
result_credit_score_sup_max = credit_df_copy.loc[credit_df_copy['CreditScore'] >= 7]['Approved']
moy_result_credit_score_sup_max = result_credit_score_sup_max.mean()
print(f"{moy_result_credit_score_sup_max.round(2)*100}% des demandes des individus avec un haut creditscore sont acceptées")
plt.figure(figsize=(8,6))
sns.histplot(result_credit_score_sup_max)
plt.title("Distribution des resultats des demandes des individus avec un haut CreditScore")
plt.show()

In [ ]:
# Analysons les outliers supérieurs au maximum de la distribution boxplot qui sont refusé
declined_with_credit_score_sup_max = credit_df_copy.loc[(credit_df_copy['CreditScore'] >= 7) & (credit_df_copy['Approved'] == 0) ]
declined_with_credit_score_sup_max

In [ ]:
print("Toute les entrées sont justifiable.\nNous n'avons pas d'imputation à faire à se niveau")

##### Quel est l'age des personnes avec un CreditScore de zéro?

In [ ]:
result_creditscore_zero = credit_df_copy.loc[(credit_df_copy['CreditScore'] == 0)]['Age']
print(f"Il y a {result_creditscore_zero.count()} personne avec un creditscore de zéro")
plt.figure(figsize=(8,5))
sns.histplot(result_creditscore_zero)
plt.title("Distribution des ages des personnes avec un CreditScore de zéro")
plt.show()

##### Quel est le resultat d'approbation des individus avec un creditscore de zéro?

In [ ]:
result_creditscore_zero = credit_df_copy.loc[(credit_df_copy['CreditScore'] == 0)]['Approved']
print(f"Il y a {result_creditscore_zero.count()} personne avec un creditscore de zéro")
plt.figure(figsize=(8,5))
sns.histplot(result_creditscore_zero)
plt.title("Distribution des résultats des demandes pour les personnes avec un CreditScore de zéro")
plt.show()

In [ ]:
# Analyse des personnes avec un creditscore de zéro et une demande approuvé
positive_result_creditscore_zero = credit_df_copy.loc[(credit_df_copy['CreditScore'] == 0) & (credit_df_copy['Approved'] == 1.0)]['Approved']
index_positive_result_creditscore_zero = positive_result_creditscore_zero.index
credit_df_copy.loc[index_positive_result_creditscore_zero]

In [ ]:
# Analyse des personnes avec un creditscore de zéro, une demande approuvé et n'etant pas des clients de banque
positive_result_creditscore_zero_no_bank = credit_df_copy.loc[(credit_df_copy['CreditScore'] == 0) & (credit_df_copy['Approved'] == 1.0) & (credit_df_copy['BankCustomer'] == 0)]['Approved']
index_positive_result_creditscore_zero_no_bank = positive_result_creditscore_zero_no_bank.index
credit_df_copy.loc[index_positive_result_creditscore_zero_no_bank]

In [ ]:
# Analyse des personnes avec un creditscore de zéro, une demande approuvé et sans années d'emploi 
positive_result_creditscore_zero_no_y_employed = credit_df_copy.loc[(credit_df_copy['CreditScore'] == 0) & (credit_df_copy['Approved'] == 1.0) & (credit_df_copy['YearsEmployed'] == 0)]['Approved']
index_positive_result_creditscore_zero_no_y_employed = positive_result_creditscore_zero_no_y_employed.index
credit_df_copy.loc[index_positive_result_creditscore_zero_no_y_employed]

In [ ]:
print("Il n'y a aucune valeur abérantes lors de l'analyse des valeurs de creditscore. En general un creditscore de zéro est un refus.\nToutefois, si approuvé, d'autre facteur peuvent l'expliquer")

##### Quelle est la corrélation entre le CreditScore et le résultat de la demande de crédit?

In [ ]:
# Quelle est la corrélation entre le CreditScore et l'approbation
matrice_de_correlation_creditscore = credit_df_copy[['CreditScore', 'Approved']]
corr_credit_score = matrice_de_correlation_creditscore.copy().corr()
corr_credit_score

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(corr_credit_score, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de Correlation des CreditScore et de l'approbation de crédit")
plt.show()

## Équilibrage des classes Credit

#### Séparation du dataset pour les données d'apprentissage

In [ ]:
#Separation du dataset X et Y(Input/Output)
X_credit = credit_df_final.iloc[:,: -1]
Y_credit = credit_df_final.iloc[:, -1]

#Séparation en 70 % train / 30 % test
X_train_credit, X_test_credit, Y_train_credit, Y_test_credit = train_test_split(X_credit, Y_credit, test_size=0.3, random_state=42)

#### Diagnostique du déséquilibre

In [ ]:
#Diagnostique du déséquilibre avant équilibrage
print("Diagnostique du déséquilibre de la cible à prédire avant équilibrage\n--------------------------------------------------------------------\n")
counts_before = Y_train_credit.value_counts()
ratio_before = counts_before.min()/counts_before.max()
print(counts_before)
print(f"\nLe ratio de déséquilibre de la classe minoritaire est {ratio_before:.2f}")
 

In [ ]:
#Sélection de la classe minoritaire et séparation du dataset X et Y(Input/Output)
X_credit_classe_minoritaire = credit_df_final.loc[credit_df_final['Approved'] == 1].iloc[:,: -1]
Y_credit_classe_minoritaire = credit_df_final.loc[credit_df_final['Approved'] == 1].iloc[:, -1]

#### Équilibrage classe avec SMOTE

In [ ]:
#Definition de la fonction de réquilibrage smote
categorical_features = list(range(6,20)) + list(range(20,25)) + list(range(30,32))
smote = SMOTENC(categorical_features, random_state=42)

#Application de la fonction de reéquilibrage
X_smote_credit, Y_smote_credit = smote.fit_resample(X_train_credit, Y_train_credit)
counts_after_smote = Y_smote_credit.value_counts()
ratio_after_smote = counts_after_smote.min()/counts_after_smote.max()

#Diagnostique du déséquilibre Avant/Après équilibrage
print("Diagnostique du déséquilibre de la cible à prédire après équilibrage avec SMOTE\n--------------------------------------------------------------------\n")
print(counts_after_smote)
print(f"\nLe ratio de déséquilibre de la classe minoritaire après équilibrage est maintenant {ratio_after_smote:.2f}")


##### Observations des attributs avec le plus de outliers après équilibrage avec SMOTE

In [ ]:
# On veut observer age, debt, yearsEmployed, creditScore, Income
colonne_credit_numerique_after = X_smote_credit.loc[:, ['Age', 'YearsEmployed', 'CreditScore', 'Income']]
colonne_credit_numerique_before = X_train_credit.loc[:, ['Age', 'YearsEmployed', 'CreditScore', 'Income']]

for columns in colonne_credit_numerique_after:
    plt.figure(figsize=(10,5))
    sns.boxplot(x = X_train_credit[columns])
    plt.title(f"Distribution de {columns} avant l'équilibrage")
    plt.show()
    plt.figure(figsize=(10,5))
    sns.boxplot(x = X_smote_credit[columns], color='yellow')
    plt.title(f"Distribution de {columns} après l'équilibrage")
    plt.show()

#### Équilibrage des classes avec RandomOverSampler

In [ ]:
#On va utiliser un overSampler car notre échantillon de données n'est pas grand

#Definition de la fonction de réquilibrage RandomOverSampler
ros = RandomOverSampler(random_state=42)

#Application de la fonction de reéquilibrage
X_ros_credit, Y_ros_credit = ros.fit_resample(X_train_credit, Y_train_credit)
counts_after_ros = Y_ros_credit.value_counts()
ratio_after_ros = counts_after_ros.min()/counts_after_ros.max()

#Diagnostique du déséquilibre Avant/Après équilibrage
print("Diagnostique du déséquilibre de la cible à prédire après équilibrage avec RandomOverSampler\n--------------------------------------------------------------------\n")
print(counts_after_ros)
print(f"\nLe ratio de déséquilibre de la classe minoritaire après équilibrage est maintenant {ratio_after_ros:.2f}")

##### Observations des attributs avec le plus de outliers après équilibrage avec RandomOverSampler

In [ ]:
# On veut observer age, debt, yearsEmployed, creditScore, Income
colonne_credit_numerique_after_ros = X_ros_credit.loc[:, ['Age', 'YearsEmployed', 'CreditScore', 'Income']]
colonne_credit_numerique_before = X_train_credit.loc[:, ['Age', 'YearsEmployed', 'CreditScore', 'Income']]

for columns in colonne_credit_numerique_after_ros:
    plt.figure(figsize=(10,5))
    sns.boxplot(x = X_train_credit[columns])
    plt.title(f"Distribution de {columns} avant l'équilibrage")
    plt.show()
    plt.figure(figsize=(10,5))
    sns.boxplot(x = X_ros_credit[columns], color='yellow')
    plt.title(f"Distribution de {columns} après l'équilibrage")
    plt.show()

#### Analyse de l'équilibrage

In [ ]:
# Description de la distribution des attributs avant l'équilibrage
X_train_credit.loc[:, ['Age','YearsEmployed','CreditScore', 'Income']].describe()

In [ ]:
# Description de la distribution des attributs après l'équilibrage avec SMOTE
X_smote_credit.loc[:, ['Age','YearsEmployed','CreditScore', 'Income']].describe()

In [ ]:
# Description de la distribution des attributs après l'équilibrage avec RandomOverSampler
X_ros_credit.loc[:, ['Age','YearsEmployed','CreditScore', 'Income']].describe()

## Modélisation supervisée Credit

#### Apprentissage supervisé

In [ ]:
# On cree une liste de modeles en tuples
models = []
models.append(('LR', LogisticRegression(solver='newton-cg')))
models.append(('DT', DecisionTreeClassifier()))
models.append(('RF', RandomForestClassifier()))
models.append(('KNN', KNeighborsClassifier()))
models.append(('SVM', SVC(probability=True)))
models.append(('NB', GaussianNB()))
models.append(('GB', GradientBoostingClassifier()))

# On cree une liste de metriques en tuples
metrics = []
metrics.append(('Accuracy', accuracy_score))
metrics.append(('Precision', precision_score))
metrics.append(('Recall', recall_score))
metrics.append(('F1-score', f1_score))
metrics.append(('AUC-ROC', roc_auc_score))

#Création d'un tableau comparatif des scores par modèle et par méthode de validation
df_comparaison_credit = pd.DataFrame(columns=['modele', 'metrique', 'methode-de-validation', 'score', 'score-train'])


In [ ]:
#Verification des scores par modèle et par métrique
for models_name, model in models:
    model.fit(X_ros_credit,Y_ros_credit)
    Y_prediction_credit = (model.predict_proba(X_test_credit)[:,1] >= 0.5).astype(int)
    Y_train_prediction_credit = (model.predict_proba(X_ros_credit)[:,1] >= 0.5).astype(int)
    for metrics_name, metric in metrics:
        comparaison_modele_credit_ajout = pd.DataFrame({'modele': [models_name], 'metrique': [metrics_name], 'methode-de-validation': ['70 % train / 30 % test'], 'score': [metric(Y_test_credit.values,Y_prediction_credit)*100], 'score-train': [metric(Y_ros_credit,Y_train_prediction_credit)*100]})
        df_comparaison_credit = pd.concat([df_comparaison_credit,comparaison_modele_credit_ajout], ignore_index=True)
        
    

#### Validation croisée

In [ ]:
# metriques
scoring = {
    'accuracy'  : make_scorer(accuracy_score),
    'Precision' : make_scorer(precision_score),
    'Recall'    : make_scorer(recall_score),
    'F1-score'  : make_scorer(f1_score),
    'AUC-ROC'   : make_scorer(roc_auc_score)
}

# Stratified K-Fold
five_f = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
seven_f = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)
ten_f = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
strat = [five_f, seven_f, ten_f]

for cv in strat:
    for model_name, model in models:
        pipe_credit = ImbPipeline(steps=[('RandomOverSampler',RandomOverSampler(random_state=42)),(model_name, model) ])
        validations_credit = cross_validate(
            pipe_credit, X_credit, Y_credit,
            cv = cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=-1
        )
        for metric in scoring:
            scores_credit = validations_credit[f'test_{metric}']
            score_train_credit = validations_credit[f'train_{metric}']
            comparaison_modele_credit_ajout = pd.DataFrame({'modele': [model_name], 'metrique': [metric], 'methode-de-validation': [f"Cross validation de {cv.n_splits} folds"], 'score': [scores_credit.mean()*100], 'score-train': [score_train.mean()*100]})
            df_comparaison_credit = pd.concat([df_comparaison_credit,comparaison_modele_credit_ajout], ignore_index=True)
            

#### Vérification de performance sur le Dataset des valeurs de la classe minoritaire

In [ ]:
#On creer le df de verification de la performance des modèles sur la classe minoritaire
df_comparaison_credit_classe_minoritaire = pd.DataFrame(columns=['modele', 'metrique', 'score'])


In [ ]:
#Verification des scores par modèle et par métrique
for models_name, model in models:
    Y_prediction_credit = (model.predict_proba(X_credit_classe_minoritaire)[:,1] >= 0.5).astype(int)
    Y_train_prediction_credit = (model.predict_proba(X_ros_credit)[:,1] >= 0.5).astype(int)
    for metrics_name, metric in metrics:
        comparaison_modele_credit_classe_minoritaire_ajout = pd.DataFrame({'modele': [models_name], 'metrique': [metrics_name],'score': [metric(Y_credit_classe_minoritaire.values,Y_prediction_credit)*100]})
        df_comparaison_credit_classe_minoritaire = pd.concat([df_comparaison_credit_classe_minoritaire,comparaison_modele_credit_classe_minoritaire_ajout], ignore_index=True)

In [ ]:
df_comparaison_credit_classe_minoritaire

## Évaluation & comparaison

#### Analyse du modèle Régression Logistique

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'LR']

#### Analyse du modèle Arbre de Décision

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'DT']

#### Analyse du modèle Random Forest

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'RF']

#### Analyse du modèle KNN

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'KNN']

#### Analyse du modèle SVM

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'SVM']

#### Analyse du modèle Naive Bayes

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'NB']

#### Analyse du modèle Gradient Boosting

In [ ]:
df_comparaison_credit.loc[df_comparaison_credit['modele'] == 'GB']